<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/Network_Traffic_Anomaly_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To develop a Python program that analyzes simulated network traffic, establishes a basic normal communication profile using destination frequency, connection time, protocol, and data volume, and identifies unusual outbound connections that significantly differ from the expected pattern.

**Algorithm**

Create a simulated network-traffic dataset.

Read the timestamp, destination, protocol, and data volume.

Calculate the normal destination frequency.

Determine the normal connection-time range.

Identify commonly used protocols.

Calculate the normal data-volume range.

Compare every outbound connection against the established profile.

Flag unusual destination, time, protocol, or data-volume characteristics.

Calculate an anomaly score based on the number of
deviations.

Display each flagged record with the characteristics that caused the alert.

In [1]:
# ==============================================
# NETWORK TRAFFIC ANOMALY DETECTOR
# ==============================================

import pandas as pd

# ------------------------------------------------
# 1. Simulated Network Traffic
# ------------------------------------------------

data = [
    ["09:00", "10.0.0.10", "HTTPS", 120],
    ["09:05", "10.0.0.10", "HTTPS", 150],
    ["09:10", "10.0.0.10", "HTTPS", 130],
    ["09:15", "10.0.0.20", "HTTPS", 110],
    ["09:20", "10.0.0.20", "HTTPS", 140],
    ["09:25", "10.0.0.10", "HTTPS", 160],
    ["09:30", "10.0.0.20", "HTTPS", 125],
    ["09:35", "10.0.0.10", "HTTPS", 135],

    # Unusual records
    ["02:10", "203.0.113.50", "TCP", 900],
    ["09:40", "198.51.100.25", "FTP", 750],
    ["09:45", "10.0.0.10", "HTTPS", 1200],
    ["23:55", "10.0.0.20", "HTTPS", 130]
]

df = pd.DataFrame(
    data,
    columns=[
        "Time",
        "Destination",
        "Protocol",
        "Data_MB"
    ]
)

# Convert time into hour
df["Hour"] = df["Time"].str.split(":").str[0].astype(int)

print("=" * 95)
print("                 NETWORK TRAFFIC ANOMALY DETECTOR")
print("=" * 95)

# ------------------------------------------------
# 2. Establish Basic Profile
# ------------------------------------------------

normal_destinations = (
    df["Destination"]
    .value_counts()
)

common_destinations = set(
    normal_destinations[
        normal_destinations >= 2
    ].index
)

normal_protocols = set(
    df["Protocol"].value_counts()
    .head(1)
    .index
)

# Normal connection time:
# Expected business hours = 08:00 - 18:00
normal_start = 8
normal_end = 18

# Typical data volume
data_mean = df["Data_MB"].mean()
data_std = df["Data_MB"].std()

data_limit = data_mean + data_std

# ------------------------------------------------
# 3. Display Profile
# ------------------------------------------------

print("\n" + "-" * 95)
print("                    NORMAL TRAFFIC PROFILE")
print("-" * 95)

print(
    "Common Destinations :",
    ", ".join(common_destinations)
)

print(
    "Common Protocol      :",
    ", ".join(normal_protocols)
)

print(
    "Normal Time Range    :",
    "08:00 - 18:00"
)

print(
    "Average Data Volume  :",
    round(data_mean, 2),
    "MB"
)

print(
    "High Volume Limit    :",
    round(data_limit, 2),
    "MB"
)

# ------------------------------------------------
# 4. Detect Anomalies
# ------------------------------------------------

alerts = []

for _, row in df.iterrows():

    reasons = []

    # Destination anomaly
    if row["Destination"] not in common_destinations:

        reasons.append(
            "Unusual destination"
        )

    # Time anomaly
    if (
        row["Hour"] < normal_start
        or row["Hour"] >= normal_end
    ):

        reasons.append(
            "Unusual connection time"
        )

    # Protocol anomaly
    if row["Protocol"] not in normal_protocols:

        reasons.append(
            "Unusual protocol"
        )

    # Data volume anomaly
    if row["Data_MB"] > data_limit:

        reasons.append(
            "Unusually high data volume"
        )

    # Create alert
    if reasons:

        alerts.append({
            "Time": row["Time"],
            "Destination": row["Destination"],
            "Protocol": row["Protocol"],
            "Data_MB": row["Data_MB"],
            "Anomaly_Score": len(reasons),
            "Reason": "; ".join(reasons)
        })

# ------------------------------------------------
# 5. Display Anomalies
# ------------------------------------------------

print("\n" + "=" * 95)
print("                    FLAGGED OUTBOUND ACTIVITY")
print("=" * 95)

if alerts:

    alert_df = pd.DataFrame(alerts)

    print(
        alert_df.to_string(
            index=False
        )
    )

else:

    print(
        "No unusual outbound activity detected."
    )

# ------------------------------------------------
# 6. Risk Classification
# ------------------------------------------------

if alerts:

    print("\n" + "=" * 95)
    print("                    ANOMALY ASSESSMENT")
    print("=" * 95)

    for _, row in alert_df.iterrows():

        score = row["Anomaly_Score"]

        if score >= 3:
            risk = "HIGH"

        elif score == 2:
            risk = "MEDIUM"

        else:
            risk = "LOW"

        print(
            f"{row['Time']} | "
            f"{row['Destination']} | "
            f"Score: {score} | "
            f"Risk: {risk}"
        )

# ------------------------------------------------
# 7. Summary
# ------------------------------------------------

print("\n" + "=" * 95)
print("                         SUMMARY")
print("=" * 95)

print(
    "Total Connections :",
    len(df)
)

print(
    "Flagged Connections :",
    len(alerts)
)

if alerts:

    print(
        "High Risk :",
        sum(
            x >= 3
            for x in alert_df["Anomaly_Score"]
        )
    )

    print(
        "Medium Risk :",
        sum(
            x == 2
            for x in alert_df["Anomaly_Score"]
        )
    )

    print(
        "Low Risk :",
        sum(
            x == 1
            for x in alert_df["Anomaly_Score"]
        )
    )

print("\nNetwork traffic analysis completed.")
print("=" * 95)

                 NETWORK TRAFFIC ANOMALY DETECTOR

-----------------------------------------------------------------------------------------------
                    NORMAL TRAFFIC PROFILE
-----------------------------------------------------------------------------------------------
Common Destinations : 10.0.0.20, 10.0.0.10
Common Protocol      : HTTPS
Normal Time Range    : 08:00 - 18:00
Average Data Volume  : 337.5 MB
High Volume Limit    : 719.77 MB

                    FLAGGED OUTBOUND ACTIVITY
 Time   Destination Protocol  Data_MB  Anomaly_Score                                                                                     Reason
02:10  203.0.113.50      TCP      900              4 Unusual destination; Unusual connection time; Unusual protocol; Unusually high data volume
09:40 198.51.100.25      FTP      750              3                          Unusual destination; Unusual protocol; Unusually high data volume
09:45     10.0.0.10    HTTPS     1200              1         

**Result**

The Python program successfully established a basic network-traffic profile and compared individual outbound connections against it. It identified anomalies based on unusual destinations, connection times, protocols, and data volumes. The most significant simulated anomaly was the 02:10 connection to 203.0.113.50, which differed from the expected pattern in all four characteristics and therefore received a HIGH anomaly assessment. The program provides the specific characteristics responsible for each alert rather than automatically declaring the traffic malicious.